In [4]:
from glob import glob
import wave
import struct
import numpy as np 

input_folder = '../Sound FX/WAV/'
output_folder = '../Sound FX/ByteData/'

In [5]:
def get_wave_samples(filepath):
    with wave.open(filepath, 'rb') as wav_file:
        nchannels = wav_file.getnchannels()
        sampwidth = wav_file.getsampwidth()
        framerate = wav_file.getframerate()
        nframes = wav_file.getnframes()
        
        # Read all frames as bytes
        raw_data = wav_file.readframes(nframes)
        
        # Determine format based on sample width (1, 2, or 4 bytes)
        # '<' indicates little-endian byte order
        if sampwidth == 1:
            fmt = f'<{nframes * nchannels}B'
        elif sampwidth == 2:
            fmt = f'<{nframes * nchannels}h'
        elif sampwidth == 4:
            fmt = f'<{nframes * nchannels}l'
        else:
            raise ValueError("Unsupported sample width")
            
        # Unpack bytes into integers
        samples = struct.unpack(fmt, raw_data)
        
    return samples, framerate
    

In [6]:
InputList = glob(input_folder+"*.wav")
for filepath in InputList:
    filename = filepath.split('/')[-1].replace('.wav','')
    samples,framerate = get_wave_samples(filepath)
    np_samples = np.array(samples)
    np_samples_normalized = (np_samples*(1/32767)*127)
    np_samples_int8 = np_samples_normalized.astype(np.int8)
    raw_bytes = np_samples_int8.tobytes()
    with open(output_folder+filename+"_"+str(framerate)+"_.dat", "wb") as file:
        file.write(raw_bytes)